# **Python OrderedDict**


In [ ]:
from collections import OrderedDict
from abc import abstractmethod

class Jury(OrderedDict):
    def __init__(self):
        super().__init__()

        for k, v in self.__class__.__dict__.items():
            if isinstance(v, type) and issubclass(v, Human):
                self[k] = v()

    @abstractmethod
    def execute(self):
        pass

class Human:
    @abstractmethod
    def reason(self):
        """ State death reason. """
        pass

    def __init_subclass__(cls, name: str, color: str, *args, **kwargs) -> None:
        print(f"Registered subclass: {cls.__qualname__} with color: {color}")
        return super().__init_subclass__(*args, **kwargs)

class Today(Jury):
    class Jade(Human, name="Jade", color="Red"):
        def reason(self): return "Sleeps in class."

    class Kenobi(Human, name="Kenobi", color="Yellow"):
        def reason(self): return "Mentally Challenged."

    def execute(self):
        if len(self) == 0: raise ValueError("No jurors available to execute.")

        for k, v in self.items():
            print(f"Human {k} with reason: {v.reason()} is going to be executed.")

today = Today()
today.execute()

# **Markov and Page Rankers**


In [ ]:
import numpy as np

web_graph = {
    # Home links to all others (hub)
    "Home":   ["Search", "News", "Blog", "Search"],  # Search appears twice

    # Search favors News, occasionally goes back to Home
    "Search": ["News", "News", "Home"],

    # News mostly links to Blog, but also to Home and Search
    "News":   ["Blog", "Blog", "Home", "Search"],

    # Blog loops on itself and sometimes goes back to Home
    "Blog":   ["Blog", "Home", "Blog"],
}

class Ranker:
    """ Google PageRanke.

    Build the Google matrix M of size N x N:

        M[i, j] = probability of transitioning from page j to page i

    (column-stochastic form).

    i = outlinks
    j = inlinks

    Runs by cols to ensure column-stochastic form.
    """

    def __init__(self, graph: dict, damping_factor: float = 0.85, epsilon: float = 1e-10):
        self.d = damping_factor
        self.epsilon = epsilon
        self.reset(graph)

    def reset(self, graph: dict):
        if not hasattr(self, "nodes"):
            self._nodes = set()
        if not hasattr(self, "graph"):
            self.graph = {}

        for node, outlinks in graph.items():
            for outlink in outlinks:
                if outlink not in self.graph:
                    self.graph.setdefault(node, []).append(outlink)
                if node not in self._nodes:
                    self._nodes.add(node)

        self.N = len(self._nodes)
        self.tag2idx = {tag: idx for idx, tag in enumerate(self.nodes)}
        self.idx2tag = {idx: tag for idx, tag in enumerate(self.nodes)}
        if not hasattr(self, 'adj'):
            self.adj = np.zeros((self.N, self.N))

    @property
    def M(self):
        """ Return the Google matrix M. """
        M = np.zeros_like(self.adj)
        for j in range(self.N):
            col = self.adj[:, j].copy()
            col_sum = np.sum(col)

            if col_sum == 0:
                # dangling node: set self-loop to 1, rest to 0
                M[:, j] = 0
                M[j, j] = 1.0
            elif col_sum != 1:
                # normalize column to sum to 1
                M[:, j] = col / col_sum
        return M

    def pageRank(self, max_iter: int | None = None):
        """ Compute PageRank vector using power iteration method. """

        M_hat = self.d * self.M
        w = np.ones(self.N) / self.N
        v = M_hat @ w + self.teleport

        c = 0
        while np.linalg.norm(w - v) >= self.epsilon:
            w = v
            v = M_hat @ w + self.teleport

            if max_iter is not None:
                max_iter -= 1

                if max_iter <= 0:
                    print("Reached maximum iterations. M never converged. Final norm:", np.linalg.norm(w - v), "Output", v)
                    return v
            c += 1

        print(f"\n{'*'*10} Converged after {c} iterations {'*'*10}\n")
        return v

    @property
    def nodes(self):
        return sorted(self._nodes, reverse=False)

    @property
    def teleport(self):
        return (1.0 - self.d) / self.N

    def update(self, graph: dict):
        """ Update adjacency matrix based on new graph structure.

        Build the Google matrix M of size N x N:

            M[i, j] = probability of transitioning from page j to page i

        (column-stochastic form).

        """

        self.reset(graph)

        missed = {}
        for key, outlinks in graph.items():
            i = self.tag2idx.get(key, None)

            if i is None:
                print(f"Key {key} not found in tag2idx.")
                missed[key] = outlinks
            else:
                for outlink in outlinks:
                    j = self.tag2idx.get(outlink, None)
                    self.adj[j, i] += 1.0

        #print(f"New adjacency matrix after update: {self.adj}")
        print(f"Missed entries during update: {missed}")
        return missed

    def predict(self, current: str):
        """
        Given a current node, return probabilities of going to all nodes
        in one step according to the transition matrix.
        """
        if current not in self.tag2idx:
            print("Queried key not in graph:", current)
            return None

        j = self.tag2idx[current]
        outlinks = self.M[:, j].copy()
        prob = {self.idx2tag[j]: outlinks[j] for j in range(len(outlinks))}
        return sorted(prob.items(), key=lambda x: x[1], reverse=True)

model = Ranker(web_graph)
model.update(web_graph)
print(model.predict("Search"))
# print(model.predictOnPowerIteration("Search"))
model.update({"Search": ["Blog", "Blog", "Home", "Home", "News"]})
print(model.predict("Search"))
print(model.pageRank())

Missed entries during update: {}
[('News', 0.6666666666666666), ('Home', 0.3333333333333333), ('Blog', 0.0), ('Search', 0.0)]
Missed entries during update: {}
[('Home', 0.375), ('News', 0.375), ('Blog', 0.25), ('Search', 0.0)]

********** Converged after 20 iterations **********

[0.43485672 0.24676879 0.14515279 0.1732217 ]


In [55]:
model.predict("Search")

[('Home', 0.375), ('News', 0.375), ('Blog', 0.25), ('Search', 0.0)]

In [20]:
model.M

array([[0.66666667, 0.25      , 0.5       , 0.        ],
       [0.33333333, 0.        , 0.25      , 0.33333333],
       [0.        , 0.25      , 0.        , 0.66666667],
       [0.        , 0.5       , 0.25      , 0.        ]])

In [103]:
for j in range(model.N):
    col = model.adj[:, j].copy()
    col_sum = np.sum(col)

    if col_sum == 0:
        # dangling node: set self-loop to 1, rest to 0
        model.adj[:, j] = 0
        model.adj[j, j] = 1.0
    elif col_sum != 1:
        # normalize column to sum to 1
        model.adj[:, j] = col / col_sum

print(f"\nNormalized M: \n\n{model.adj}")

M_hat = model.d * model.adj
w = np.ones(model.N) / model.N
w[3] += 1.0  # initial rank vector e.g. what the user has previously visited or done or what we already know about them
w /= w.sum() # asserts to sum to 1
v = M_hat @ w + (1 - model.d) / model.N
print(f"\nw: \n{w}\n\nInitial norm: {np.linalg.norm(w - v)}\n\nM_hat:\n\n{M_hat}\n\nInitial v: \n{v}")
#print(f"{'*'*10} Iteration: {'*'*10}\n")

c = 0
while np.linalg.norm(w - v) >= model.epsilon:
    w = v
    v = M_hat @ w + (1 - model.d) / model.N
    c+=1
    #print(v)


Normalized M: 

[[0.46720236 0.36492891 0.         0.        ]
 [0.18694654 0.         0.28578591 0.64912281]
 [0.3458511  0.27014218 0.         0.35087719]
 [0.         0.36492891 0.71421409 0.        ]]

w: 
[0.125 0.125 0.125 0.625]

Initial norm: 0.5875315308513221

M_hat:

[[0.39712201 0.31018957 0.         0.        ]
 [0.15890456 0.         0.24291802 0.55175439]
 [0.29397343 0.22962085 0.         0.29824561]
 [0.         0.31018957 0.60708198 0.        ]]

Initial v: 
[0.12591395 0.43257431 0.28935279 0.15215894]


In [93]:
model.nodes

['Blog', 'Home', 'News', 'Search']

In [39]:
import numpy as np

def pagerank(M, d: float = 0.85, eps: float = 1e-10):
    """PageRank algorithm with explicit number of iterations. Returns ranking of nodes (pages) in the adjacency matrix.

    Parameters
    ----------
    M : numpy array
        adjacency matrix where M_i,j represents the link from 'j' to 'i', such that for all 'j'
        sum(i, M_i,j) = 1
    d : float, optional
        damping factor, by default 0.85

    Returns
    -------
    numpy array
        a vector of ranks such that v_i is the i-th rank from [0, 1],

    NOTE: When M is identity matrix, the ranks should be uniform i.e. [0.25, 0.25, 0.25, 0.25]
    """

    # Normalize the cols of M to ensure column-stochasticity. This ensures convergence
    N = M.shape[1]
    for j in range(N):
        col = M[:, j].copy()
        col_sum = np.sum(col)

        if col_sum == 0:
            # dangling node: set self-loop to 1, rest to 0
            M[:, j] = 0
            M[j, j] = 1.0
        elif col_sum != 1:
            # normalize column to sum to 1
            M[:, j] = col / col_sum

    print(f"\nNormalized M: \n\n{M}")

    M_hat = d * M
    w = np.ones(N) / N
    v = M_hat @ w + (1 - d) / N
    c = 0
    while np.linalg.norm(w - v) >= eps:
        w = v
        v = M_hat @ w + (1 - d) / N
        c+=1
        #print(v)

    print(f"\n{'*'*10} Converged after {c} iterations {'*'*10}\n")
    return v

M2 = np.array([[0, 0, 0, .25],
              [0, 0, 0, .5],
              [1, 0.5, 0, .25],
              [0, 0.5, 1, 0]])

pagerank(model.adj)


Normalized M: 

[[0.66666667 0.25       0.5        0.        ]
 [0.33333333 0.         0.25       0.33333333]
 [0.         0.25       0.         0.66666667]
 [0.         0.5        0.25       0.        ]]

********** Converged after 22 iterations **********



array([0.3905578 , 0.23950027, 0.19023036, 0.17971157])

In [96]:
# When M is identity matrix, the ranks should be uniform i.e. [0.25, 0.25, 0.25, 0.25]
M = np.array([[10, 0, 1, 0],
              [0, 0, 0, 0],
              [0, 0.65, 0.1, 0.25],
              [0, 0.4, 0.3, 0.3]])

# 1. Identity matrix (self-loops only) - should give uniform ranks
M_identity = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=float)
print("=== Identity Matrix (all self-loops) ===")
v1 = pagerank(M_identity)
print(f"Final ranks: {v1}\n")

# 2. Fully connected (everyone links to everyone)
M_full = np.ones((4, 4), dtype=float) / 4
print("=== Fully Connected (uniform transition) ===")
v2 = pagerank(M_full)
print(f"Final ranks: {v2}\n")

# 3. Hub-and-spoke (node 0 is a hub)
M_hub = np.array([
    [0, 1, 1, 1],    # everyone links to node 0
    [0.33, 0, 0, 0], # node 0 links to 1, 2, 3 equally
    [0.33, 0, 0, 0],
    [0.34, 0, 0, 0]
], dtype=float)
print("=== Hub-and-Spoke (node 0 is central hub) ===")
v3 = pagerank(M_hub)
print(f"Final ranks: {v3}\n")

# 4. Linear chain A -> B -> C -> D
M_chain = np.array([
    [0, 0, 0, 0],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0]
], dtype=float)
print("=== Linear Chain (A→B→C→D, D is dangling) ===")
v4 = pagerank(M_chain)
print(f"Final ranks: {v4}\n")

# 5. Cycle A -> B -> C -> D -> A
M_cycle = np.array([
    [0, 0, 0, 1],
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0]
], dtype=float)
print("=== Cycle (A→B→C→D→A) ===")
v5 = pagerank(M_cycle)
print(f"Final ranks: {v5}\n")

# 6. Two separate components (disconnected graph)
M_disconnected = np.array([
    [0, 1, 0, 0],
    [1, 0, 0, 0],
    [0, 0, 0, 1],
    [0, 0, 1, 0]
], dtype=float)
print("=== Disconnected (A↔B, C↔D, no bridge) ===")
v6 = pagerank(M_disconnected)
print(f"Final ranks: {v6}\n")

# 7. Authority-Hub structure (bipartite-like)
M_authority = np.array([
    [0, 0.5, 0.5, 0],
    [0.5, 0, 0, 0.5],
    [0.5, 0, 0, 0.5],
    [0, 0.5, 0.5, 0]
], dtype=float)
print("=== Authority-Hub (nodes 0,3 are hubs; 1,2 are authorities) ===")
v7 = pagerank(M_authority)
print(f"Final ranks: {v7}\n")

# 8. One supernode (node 0 receives all links, distributes uniformly)
M_supernode = np.array([
    [0, 1, 1, 1],
    [0.33, 0, 0, 0],
    [0.33, 0, 0, 0],
    [0.34, 0, 0, 0]
], dtype=float)
print("=== Supernode (all point to 0, 0 distributes uniformly) ===")
v8 = pagerank(M_supernode)
print(f"Final ranks: {v8}\n")

# 9. Sparse with dangling nodes
M_sparse_dangling = np.array([
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 0],  # dangling node (no outlinks)
    [0, 0, 0, 0]   # dangling node
], dtype=float)
print("=== Sparse with Dangling Nodes (nodes 2,3 are dangling) ===")
v9 = pagerank(M_sparse_dangling)
print(f"Final ranks: {v9}\n")

# 10. Random sparse matrix
np.random.seed(42)
M_random = np.random.rand(4, 4)
M_random[M_random < 0.7] = 0  # make it sparse
print("=== Random Sparse Matrix ===")
v10 = pagerank(M_random)
print(f"Final ranks: {v10}\n")


=== Identity Matrix (all self-loops) ===

Normalized M: 

[[1. 0. 0. 0.]
 [0. 1. 0. 0.]
 [0. 0. 1. 0.]
 [0. 0. 0. 1.]]

Initial norm: 0.0649519052838329

M_hat:

[[0.85 0.   0.   0.  ]
 [0.   0.85 0.   0.  ]
 [0.   0.   0.85 0.  ]
 [0.   0.   0.   0.85]]

Initial v: 
[0.56875 0.14375 0.14375 0.14375]

********** Converged after 125 iterations **********

Final ranks: [0.25 0.25 0.25 0.25]

=== Fully Connected (uniform transition) ===

Normalized M: 

[[0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]
 [0.25 0.25 0.25 0.25]]

Initial norm: 0.4330127018922193

M_hat:

[[0.2125 0.2125 0.2125 0.2125]
 [0.2125 0.2125 0.2125 0.2125]
 [0.2125 0.2125 0.2125 0.2125]
 [0.2125 0.2125 0.2125 0.2125]]

Initial v: 
[0.25 0.25 0.25 0.25]

********** Converged after 1 iterations **********

Final ranks: [0.25 0.25 0.25 0.25]

=== Hub-and-Spoke (node 0 is central hub) ===

Normalized M: 

[[0.   1.   1.   1.  ]
 [0.33 0.   0.   0.  ]
 [0.33 0.   0.   0.  ]
 [0.34 0.   0.   0.  ]]

Ini